# Exploring the telematics data

This notebook takes a first look at `data/example_telematics_data.csv`: a single vehicle's trip with **vehicle speed** (km/h) and **engine speed** (rpm) over time.

The charts use [plotly.express](https://plotly.com/python/plotly-express/), so you can zoom, pan and hover to inspect values.

In [8]:
import pandas as pd
import plotly.express as px

from ds_dagster_intro.defs.processing import interpolate_vehicle_speed, load_telematics

In [9]:
df = load_telematics("../data/example_telematics_data.csv")
df.head(10)

,serial,timestamp,vehicle_speed_kmph,engine_speed_rpm
0,abc-1234,2026-09-17 19:14:24+00:00,0.0,699.375
1,abc-1234,2026-09-17 19:14:25+00:00,NaN,700.625
2,abc-1234,2026-09-17 19:14:26+00:00,NaN,699.750
3,abc-1234,2026-09-17 19:14:27+00:00,NaN,699.500
4,abc-1234,2026-09-17 19:14:28+00:00,NaN,699.500
5,abc-1234,2026-09-17 19:14:29+00:00,0.0,699.750
6,abc-1234,2026-09-17 19:14:30+00:00,NaN,699.875
7,abc-1234,2026-09-17 19:14:31+00:00,NaN,701.125
8,abc-1234,2026-09-17 19:14:32+00:00,NaN,700.250
9,abc-1234,2026-09-17 19:14:33+00:00,NaN,700.125


In [10]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 30859 entries, 0 to 30858
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype              
---  ------              --------------  -----              
 0   serial              30859 non-null  str                
 1   timestamp           30859 non-null  datetime64[us, UTC]
 2   vehicle_speed_kmph  6172 non-null   float64            
 3   engine_speed_rpm    30858 non-null  float64            
dtypes: datetime64[us, UTC](1), float64(2), str(1)
memory usage: 964.5 KB


## Reporting rates

Engine speed is reported every **1 second**, but vehicle speed only every **5 seconds**. That's why most vehicle speed rows are `NaN`. The `preprocessed_telematics` asset fills these gaps by interpolation.

In [11]:
print("Missing values per column:")
print(df.isna().sum())
print()
print("Time between consecutive rows:")
print(df["timestamp"].diff().value_counts())

Missing values per column:
serial                    0
timestamp                 0
vehicle_speed_kmph    24687
engine_speed_rpm          1
dtype: int64

Time between consecutive rows:
timestamp
0 days 00:00:01    30847
0 days 00:00:02        9
0 days 00:00:24        1
0 days 00:00:03        1
Name: count, dtype: int64


## Full trip

In [12]:
long = df.melt(
    id_vars="timestamp",
    value_vars=["vehicle_speed_kmph", "engine_speed_rpm"],
    var_name="sensor",
    value_name="value",
).dropna()

fig = px.scatter(
    long,
    x="timestamp",
    y="value",
    facet_row="sensor",
    color="sensor",
    height=600,
    title="Vehicle speed and engine speed over the trip",
)
fig.update_yaxes(matches=None, title_text="")
fig.update_layout(showlegend=False)
fig.show()

## Zooming in: gaps and interpolation

Here is a 2-minute window. The squares show the raw vehicle speed samples (one every 5 s), and the circles show the interpolated values from the pipeline's preprocessing function.

In [13]:
interpolated = interpolate_vehicle_speed(df)

start = df.loc[df["vehicle_speed_kmph"] > 20, "timestamp"].iloc[0]
window = (df["timestamp"] >= start) & (df["timestamp"] < start + pd.Timedelta(minutes=2))

fig = px.scatter(
    interpolated[window],
    x="timestamp",
    y="vehicle_speed_kmph",
    title="Vehicle speed: raw samples vs interpolated",
    labels={"vehicle_speed_kmph": "vehicle speed (km/h)"},
)
fig.add_scatter(
    x=df.loc[window, "timestamp"],
    y=df.loc[window, "vehicle_speed_kmph"],
    mode="markers",
    marker_symbol="square",
    name="raw samples",
)
fig.data[0].name = "interpolated"
fig.data[0].showlegend = True
fig.show()

## Vehicle speed vs engine speed

In [14]:
fig = px.scatter(
    interpolated,
    x="engine_speed_rpm",
    y="vehicle_speed_kmph",
    opacity=0.2,
    title="Vehicle speed vs engine speed",
    labels={"engine_speed_rpm": "engine speed (rpm)", "vehicle_speed_kmph": "vehicle speed (km/h)"},
)
fig.show()

## Your observations

What patterns or data quality issues do you notice? Note anything you'd want to investigate further.